# 10 — Runtime topology design

**Per-request topology synthesis. Closes the loop between offline ADAS (notebook 08) and the existing runtime agent design (`MetaOrchestrator`).**

`MetaOrchestrator` already does runtime LLM-driven planning, but its output is a *flat* `TaskDecomposition` (a list of subtasks). It can't emit branching, parallelism, refinement loops, or routing. This notebook ships the missing piece: a per-request designer that emits a typed `TopologySpec` and a `TopologyOrchestrator` that hydrates and runs it.

What you'll see:

1. Cold runtime design — one LLM call → one `TopologySpec` → live runtime.
2. `MemoryStoreCache` — second similar request hits the cache (online learning over "this topology worked for goals like that").
3. `TopologyOrchestrator` end-to-end — design + execute + record outcome in one call.
4. Seed library — a small library of validated topologies passed to the designer biases it toward known-good shapes.
5. Online learning — a failure decays the cached entry's reliability; the next similar request re-designs.

Cost: ~\$1–2 against the project's default model.


## 1. Setup — agents + registries + designer agent

In [1]:
from __future__ import annotations

from typing import Any
from pydantic import BaseModel, Field

from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()
print(f"Task model: {config.llm_model}")


class Answer(BaseModel):
    answer: str = Field(description="Answer to the question.")


class CotAgent(BaseAgent[GlobalState, Answer]):
    async def _run_implementation(self, state, **kw):
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output


def _make_factory(name: str, prompt: str):
    def _f():
        return CotAgent(
            agent_name=name,
            system_prompt=prompt,
            output_type=Answer,
            model=config.llm_model,
            api_key=config.llm_api_key,
        )
    return _f


agent_registry = {
    "cot":      _make_factory("cot",      "You answer questions concisely. If unsure, say so."),
    "verifier": _make_factory("verifier", "Re-check the candidate answer for correctness given the paragraph; correct if wrong; abstain if unanswerable."),
    "summarizer": _make_factory("summarizer", "Summarize the input in one sentence."),
}
print("agents:", sorted(agent_registry))


Task model: openai:gpt-5.2
agents: ['cot', 'summarizer', 'verifier']


In [2]:
# The designer agent — a BaseAgent whose structured output is a TopologyDesign.
# Pydantic-AI handles the JSON schema for TopologySpec automatically.
from orqest.optimization import TopologyDesign


class DesignerAgent(BaseAgent[GlobalState, TopologyDesign]):
    async def _run_implementation(self, state, **kw):
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output


designer_agent = DesignerAgent(
    agent_name="designer",
    system_prompt=(
        "You design typed Orqest topologies (Pipeline / Parallel / Router / RefinementLoop) "
        "for incoming goals. Always emit a `thought` and a `spec`. Never emit raw Python. "
        "Use ONLY agent_name and callable_name values from the user message's allowlist; "
        "hydration will fail and your design will be discarded if you reference unknown names."
    ),
    output_type=TopologyDesign,
    model=config.llm_model,
    api_key=config.llm_api_key,
)
print("designer ready")


designer ready


## 2. Cold runtime design — no cache

In [3]:
import asyncio
from orqest.autonomy.runtime import NoCache, RuntimeTopologyDesigner
from orqest.orchestration.hydrate import CallableRegistry

callable_registry = CallableRegistry()  # empty — designer must use only agents

cold_designer = RuntimeTopologyDesigner(
    designer_agent=designer_agent,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
    constraints="Prefer the simplest topology that fits. Add a verifier step only when the goal explicitly mentions 'verify' or 'check'.",
    cache=NoCache(),
)

import time
t0 = time.monotonic()
goal = "Summarize the following paragraph and verify your summary stays faithful: 'Python was first released in 1991 by Guido van Rossum.'"
spec = await cold_designer.design(goal)
design_ms = (time.monotonic() - t0) * 1000
print(f"design_ms: {design_ms:.0f}")
print(f"spec.kind: {spec.kind}")
print(f"spec JSON:")
print(spec.model_dump_json(indent=2)[:600])


design_ms: 4310
spec.kind: pipeline
spec JSON:
{
  "kind": "pipeline",
  "steps": [
    {
      "operation": {
        "kind": "agent_step",
        "agent_name": "summarizer",
        "inline_spec": null
      },
      "config": null
    },
    {
      "operation": {
        "kind": "agent_step",
        "agent_name": "verifier",
        "inline_spec": null
      },
      "config": null
    }
  ],
  "name": "summarize_then_verify"
}


## 3. With `MemoryStoreCache` — second similar request hits the cache

We use the project's default sentence-transformers embedder via `LocalMemoryStore`. Every successful design is stored as a semantic memory in the `topology_cache` namespace; subsequent semantically-similar goals retrieve the prior spec without paying for another design call.

In [4]:
from orqest.memory import LocalMemoryStore
from orqest.autonomy.runtime import MemoryStoreCache


# Embedder — simple character-overlap proxy is fine for the demo (real prod would use a sentence-transformer)
async def fake_embedder(text: str) -> list[float]:
    """Tiny demo embedder: maps to a 64-dim vector by char hashing."""
    import hashlib
    h = hashlib.sha256(text.lower().encode()).digest()[:64]
    return [b / 255.0 for b in h]


store = LocalMemoryStore(":memory:", embedder=fake_embedder)
cache = MemoryStoreCache(store, threshold=0.0, namespace="topology_cache")  # threshold 0 → always match top hit

cached_designer = RuntimeTopologyDesigner(
    designer_agent=designer_agent,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
    constraints="Prefer the simplest topology that fits.",
    cache=cache,
)

goals = [
    "Summarize the input paragraph.",
    "Briefly summarize this paragraph.",
    "Give me a one-sentence summary of this text.",
]
for g in goals:
    t0 = time.monotonic()
    spec = await cached_designer.design(g)
    elapsed = (time.monotonic() - t0) * 1000
    print(f"  goal: {g!r:<60} → kind={spec.kind:<10} {elapsed:>6.0f}ms")

print()
print(f"stats: lookups={cached_designer.stats.lookups} hits={cached_designer.stats.hits} misses={cached_designer.stats.misses} designs={cached_designer.stats.designs}")


  goal: 'Summarize the input paragraph.'                             → kind=pipeline     3550ms
  goal: 'Briefly summarize this paragraph.'                          → kind=pipeline        4ms
  goal: 'Give me a one-sentence summary of this text.'               → kind=pipeline        3ms

stats: lookups=3 hits=2 misses=1 designs=1


## 4. `TopologyOrchestrator` — full design + execute loop

Wraps the designer with the design-hydrate-run-record loop. Each call returns a `TopologyExecutionResult` with timing, structural metrics, and cache-hit signal.

In [5]:
from orqest.autonomy.topology_orchestrator import TopologyOrchestrator

orchestrator = TopologyOrchestrator(
    cached_designer,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
)

goals_to_run = [
    "Summarize this paragraph: 'Python was first released in 1991.'",
    "What's a brief summary of: 'JavaScript first appeared in 1995.'",
    "Verify and answer concisely: 'Tesla, founded in 2003, has Elon Musk as CEO.' Question: who is the CEO?",
]
results = []
for g in goals_to_run:
    r = await orchestrator.execute(g)
    results.append(r)

print(f"{'goal':<60} {'kind':<10} {'cache':<6} {'design_ms':>9} {'exec_ms':>9}")
print("-" * 110)
for r in results:
    g_short = r.goal[:55] + "..." if len(r.goal) > 58 else r.goal
    print(f"{g_short:<60} {r.spec_kind:<10} {str(r.cache_hit):<6} {r.design_ms:>9.0f} {r.execution_ms:>9.0f}")


goal                                                         kind       cache  design_ms   exec_ms
--------------------------------------------------------------------------------------------------------------
Summarize this paragraph: 'Python was first released in...   pipeline   True           4      1487
What's a brief summary of: 'JavaScript first appeared i...   pipeline   True           3      1326
Verify and answer concisely: 'Tesla, founded in 2003, h...   pipeline   True           3      1326


## 5. Seed library — biasing the designer toward known-good topologies

In production you'd typically run `MetaAgentSearch` (notebook 08) offline against a representative gold set, then pass the Pareto-front topologies as a seed library at runtime. The designer prefers composing or specializing them over inventing fresh structures.

In [6]:
from orqest.orchestration.spec import (
    AgentStepSpec, PipelineSpec, PipelineStepEntry,
)

seed_library = [
    # Known good: a CoT → Verifier 2-step pipeline for verify-style goals
    PipelineSpec(
        name="cot_then_verify",
        steps=[
            PipelineStepEntry(operation=AgentStepSpec(agent_name="cot")),
            PipelineStepEntry(operation=AgentStepSpec(agent_name="verifier")),
        ],
    ),
    # Known good: a single Summarizer for summarize-style goals
    PipelineSpec(
        name="single_summarizer",
        steps=[PipelineStepEntry(operation=AgentStepSpec(agent_name="summarizer"))],
    ),
]

seeded_designer = RuntimeTopologyDesigner(
    designer_agent=designer_agent,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
    constraints="Prefer the simplest topology. Use the library entries when they fit.",
    cache=NoCache(),  # disable cache so we see the designer's seeded output
    seed_library=seed_library,
)

goal = "Summarize and then verify: 'Berlin, the capital of Germany, has 3.7 million people.'"
spec = await seeded_designer.design(goal)
print(f"designed kind: {spec.kind}")
print(f"spec JSON:")
print(spec.model_dump_json(indent=2)[:800])


designed kind: pipeline
spec JSON:
{
  "kind": "pipeline",
  "steps": [
    {
      "operation": {
        "kind": "agent_step",
        "agent_name": "summarizer",
        "inline_spec": null
      },
      "config": null
    },
    {
      "operation": {
        "kind": "agent_step",
        "agent_name": "verifier",
        "inline_spec": null
      },
      "config": null
    }
  ],
  "name": "summarize_then_verify"
}


## 6. Online learning — failure decays a cached entry

If a hydrated topology runs but raises an exception, the orchestrator records a failure to the cache. The existing memory machinery decays the entry's reliability; with a strict `min_reliability` filter, future similar requests skip the bad cached entry and re-design.

In [7]:
# Build a fresh cache + designer to see the decay clearly
decay_store = LocalMemoryStore(":memory:", embedder=fake_embedder)
decay_cache = MemoryStoreCache(decay_store, threshold=0.0, min_reliability=0.0, namespace="topology_cache")

# Seed the cache with a topology that references a non-existent agent
from orqest.orchestration.spec import PipelineStepEntry, AgentStepSpec, PipelineSpec
bad_spec = PipelineSpec(steps=[PipelineStepEntry(operation=AgentStepSpec(agent_name="ghost"))])
await decay_cache.store("do something", bad_spec, success=True)
print("cached bad spec; lookup OK?", (await decay_cache.lookup("do something")) is not None)

decay_designer = RuntimeTopologyDesigner(
    designer_agent=designer_agent,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
    cache=decay_cache,
    verify_on_hit=False,  # disabled so the bad cache hit reaches execution
)
decay_orch = TopologyOrchestrator(
    decay_designer,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
)

# Execute — will fail because 'ghost' isn't in agent_registry
try:
    await decay_orch.execute("do something")
except Exception as exc:
    print(f"execution failed (expected): {type(exc).__name__}")

# Now use a strict min_reliability filter — the decayed entry should NOT be returned
strict_cache = MemoryStoreCache(decay_store, threshold=0.0, min_reliability=0.9, namespace="topology_cache")
recalled = await strict_cache.lookup("do something")
print(f"strict filter recall after decay: {recalled}")  # None


cached bad spec; lookup OK? True
execution failed (expected): KeyError
strict filter recall after decay: None


## What's next

- **W3.C — sandboxed codegen.** When the meta agent emits raw Python `forward()` for cases where compositions of registered primitives are too constrained.
- **W3.D — RetrievalPolicy Protocol over the seed library.** Today `seed_library` is a flat list passed to the designer's prompt. A Protocol layer would let the designer fetch from a vector store / external API based on goal similarity instead of holding the whole library in the prompt.
- **W3.E — Output-quality reliability signal.** Today decay only fires on execution exceptions. Wiring an `output_judge` callable into `TopologyOrchestrator.execute` would let consumers decay topologies that produce *bad outputs* (not just crashing ones).
- **W3.F — `MetacognitionHook` integration.** Surface per-step confidence into `TopologyExecutionResult` so cache reliability can also reflect "ran fine but agent reported low confidence."

For the design rationale, see the **Runtime topology design** section in `docs/concepts/topology_optimization.md`.
